# 01 — Bona Data Analysis
## SunnyBest Retail Forecasting System — Raw Database Tables

Analyses all source tables pulled directly from the Supabase PostgreSQL database (`core` schema).

**Dimension tables:** `dim_calendar`, `dim_policy_regimes`, `dim_products`, `dim_stores`

**Fact tables:** `fact_sales`, `fact_inventory`, `fact_customer_activity`, `fact_store_operations`, `fact_promotions`, `fact_restriction_events`, `fact_weather`

---
## 0. DB Connection & Load All Tables

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from sqlalchemy import create_engine
from urllib.parse import quote_plus

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# ------- Connection (fill in your credentials) -------
host     = "aws-1-eu-central-1.pooler.supabase.com"
port     = 5432
database = "postgres"
user     = "postgres.ogkdfmkybqtrsglcizzt"
password = quote_plus("YOUR_PASSWORD")   # replace with your password

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}",
    pool_pre_ping=True
)

print("Engine created.")

In [ ]:
# -------------------------
# DIM TABLES
# -------------------------
df_calendar = pd.read_sql("SELECT * FROM core.dim_calendar ORDER BY date ASC", engine)
df_policy_regimes = pd.read_sql("SELECT * FROM core.dim_policy_regimes ORDER BY start_date ASC", engine)
df_products = pd.read_sql("SELECT * FROM core.dim_products ORDER BY product_id ASC", engine)
df_stores = pd.read_sql("SELECT * FROM core.dim_stores ORDER BY store_id ASC", engine)

# -------------------------
# FACT TABLES
# -------------------------
df_sales = pd.read_sql("SELECT * FROM core.fact_sales ORDER BY date ASC", engine)
df_inventory = pd.read_sql("SELECT * FROM core.fact_inventory ORDER BY date ASC", engine)
df_customer_activity = pd.read_sql("SELECT * FROM core.fact_customer_activity ORDER BY date ASC", engine)
df_store_operations = pd.read_sql("SELECT * FROM core.fact_store_operations ORDER BY date ASC", engine)
df_promotions = pd.read_sql("SELECT * FROM core.fact_promotions ORDER BY date ASC", engine)
df_restrictions = pd.read_sql("SELECT * FROM core.fact_restriction_events ORDER BY date ASC", engine)
df_weather = pd.read_sql("SELECT * FROM core.fact_weather ORDER BY date ASC", engine)

print("All tables loaded successfully.")

---
## 1. Dataset Overview — All Tables at a Glance

In [ ]:
all_tables = {
    "dim_calendar":           df_calendar,
    "dim_policy_regimes":     df_policy_regimes,
    "dim_products":           df_products,
    "dim_stores":             df_stores,
    "fact_sales":             df_sales,
    "fact_inventory":         df_inventory,
    "fact_customer_activity": df_customer_activity,
    "fact_store_operations":  df_store_operations,
    "fact_promotions":        df_promotions,
    "fact_restriction_events":df_restrictions,
    "fact_weather":           df_weather,
}

rows = []
for name, d in all_tables.items():
    date_col = next((c for c in d.columns if "date" in c.lower()), None)
    date_range = f"{d[date_col].min()} → {d[date_col].max()}" if date_col else "—"
    missing_pct = round(d.isnull().sum().sum() / d.size * 100, 2)
    rows.append({
        "Table":       name,
        "Type":        "DIM" if name.startswith("dim") else "FACT",
        "Rows":        f"{len(d):,}",
        "Columns":     d.shape[1],
        "Missing %":   f"{missing_pct}%",
        "Date Range":  date_range,
    })

overview = pd.DataFrame(rows).set_index("Table")
display(overview)

In [ ]:
# Column-level missing values per table
for name, d in all_tables.items():
    missing = d.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        print(f"\n{name} — columns with nulls:")
        for col, cnt in missing.items():
            print(f"  {col:<35} {cnt:>6} ({cnt/len(d)*100:.1f}%)")
    else:
        print(f"{name} — no missing values ✓")

---
## 2. Dimension Tables

### 2a. dim_stores

In [ ]:
print(f"Stores: {len(df_stores)}")
display(df_stores)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_stores["city"].value_counts().plot(kind="bar", ax=axes[0], color="#4C72B0", rot=30)
axes[0].set_title("Stores by City")

df_stores["store_size"].value_counts().plot(kind="bar", ax=axes[1], color="#55A868", rot=0)
axes[1].set_title("Stores by Size")

plt.tight_layout()
plt.show()

### 2b. dim_products

In [ ]:
print(f"Products: {len(df_products)}")
display(df_products.head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_products["category"].value_counts().sort_values().plot(kind="barh", ax=axes[0], color="#C44E52")
axes[0].set_title("Products by Category")

df_products["brand"].value_counts().head(10).sort_values().plot(kind="barh", ax=axes[1], color="#DD8452")
axes[1].set_title("Top 10 Brands")

plt.tight_layout()
plt.show()

print(f"\nPrice range: ₦{df_products['regular_price'].min():,.0f} — ₦{df_products['regular_price'].max():,.0f}")
print(f"Avg cost price: ₦{df_products['cost_price'].mean():,.0f}")

### 2c. dim_calendar

In [ ]:
print(f"Calendar rows: {len(df_calendar)}")
print(f"Date range: {df_calendar['date'].min()} → {df_calendar['date'].max()}")
display(df_calendar.head())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df_calendar["season"].value_counts().plot(kind="bar", ax=axes[0], color="#8172B2", rot=0)
axes[0].set_title("Days by Season")

df_calendar["is_holiday"].value_counts().plot(kind="bar", ax=axes[1], color="#64B5CD", rot=0)
axes[1].set_title("Holiday Days")
axes[1].set_xticklabels(["Non-Holiday", "Holiday"])

df_calendar["is_payday"].value_counts().plot(kind="bar", ax=axes[2], color="#55A868", rot=0)
axes[2].set_title("Payday Days")
axes[2].set_xticklabels(["Non-Payday", "Payday"])

plt.tight_layout()
plt.show()

### 2d. dim_policy_regimes

In [ ]:
print(f"Policy regimes: {len(df_policy_regimes)}")
display(df_policy_regimes)

---
## 3. Fact Tables

### 3a. fact_sales

In [ ]:
print(f"fact_sales: {len(df_sales):,} rows | {df_sales['date'].min()} → {df_sales['date'].max()}")
display(df_sales.head())
display(df_sales[["units_sold", "revenue", "price", "discount_pct"]].describe().round(2))

In [ ]:
# Monthly revenue trend from fact_sales
df_sales["date"] = pd.to_datetime(df_sales["date"])
monthly_rev = df_sales.groupby(df_sales["date"].dt.to_period("M"))["revenue"].sum() / 1e6
monthly_rev.index = monthly_rev.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly_rev.index, monthly_rev.values, color="#4C72B0", linewidth=2)
ax.fill_between(monthly_rev.index, monthly_rev.values, alpha=0.15, color="#4C72B0")
ax.set_title("Monthly Revenue Trend — fact_sales (₦M)")
ax.set_ylabel("Revenue (₦M)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# Units sold distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_sales["units_sold"].clip(upper=df_sales["units_sold"].quantile(0.99)).hist(bins=50, ax=axes[0], color="#4C72B0", edgecolor="white")
axes[0].set_title("Distribution: units_sold")

(df_sales["revenue"] / 1e3).clip(upper=(df_sales["revenue"]/1e3).quantile(0.99)).hist(bins=50, ax=axes[1], color="#DD8452", edgecolor="white")
axes[1].set_title("Distribution: revenue (₦K)")
plt.tight_layout()
plt.show()

In [ ]:
# Sales by store and product
sales_by_store = df_sales.groupby("store_id").agg(
    total_units=("units_sold", "sum"),
    total_revenue=("revenue", "sum"),
    avg_price=("price", "mean"),
).sort_values("total_revenue", ascending=False)

sales_by_store["revenue_M"] = (sales_by_store["total_revenue"] / 1e6).round(1)
display(sales_by_store[["total_units", "revenue_M", "avg_price"]].rename(columns={"revenue_M": "revenue (₦M)"}))

fig, ax = plt.subplots(figsize=(10, 4))
sales_by_store["revenue_M"].sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Total Revenue by Store (₦M)")
plt.tight_layout()
plt.show()

### 3b. fact_inventory

In [ ]:
df_inventory["date"] = pd.to_datetime(df_inventory["date"])
print(f"fact_inventory: {len(df_inventory):,} rows | {df_inventory['date'].min().date()} → {df_inventory['date'].max().date()}")
display(df_inventory.head())
display(df_inventory.describe().round(2))

# Stockout rate
if "stockout_occurred" in df_inventory.columns:
    so_rate = df_inventory["stockout_occurred"].mean() * 100
    print(f"\nOverall stockout rate: {so_rate:.2f}%")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    so_store = df_inventory.groupby("store_id")["stockout_occurred"].mean().sort_values() * 100
    so_store.plot(kind="barh", ax=axes[0], color="#C44E52")
    axes[0].set_title("Stockout Rate by Store (%)")

    inv_monthly = df_inventory.groupby(df_inventory["date"].dt.to_period("M"))["starting_inventory"].mean()
    inv_monthly.index = inv_monthly.index.astype(str)
    axes[1].plot(inv_monthly.index, inv_monthly.values, color="#55A868", linewidth=2)
    axes[1].set_title("Avg Starting Inventory Over Time")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

### 3c. fact_promotions

In [ ]:
df_promotions["date"] = pd.to_datetime(df_promotions["date"])
print(f"fact_promotions: {len(df_promotions):,} rows | {df_promotions['date'].min().date()} → {df_promotions['date'].max().date()}")
display(df_promotions.head())
display(df_promotions.describe(include="all").T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if "promo_type" in df_promotions.columns:
    df_promotions["promo_type"].value_counts().plot(kind="bar", ax=axes[0], color="#8172B2", rot=20)
    axes[0].set_title("Promotions by Type")

if "discount_pct" in df_promotions.columns:
    df_promotions["discount_pct"].hist(bins=30, ax=axes[1], color="#64B5CD", edgecolor="white")
    axes[1].set_title("Distribution of Discount %")

plt.tight_layout()
plt.show()

### 3d. fact_customer_activity

In [ ]:
df_customer_activity["date"] = pd.to_datetime(df_customer_activity["date"])
print(f"fact_customer_activity: {len(df_customer_activity):,} rows")
display(df_customer_activity.head())
display(df_customer_activity.describe().round(2))

# Footfall / visits over time
num_cols = df_customer_activity.select_dtypes(include="number").columns.tolist()
fig, axes = plt.subplots(1, min(len(num_cols), 3), figsize=(16, 4))
if len(num_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, num_cols[:3]):
    monthly = df_customer_activity.groupby(df_customer_activity["date"].dt.to_period("M"))[col].mean()
    monthly.index = monthly.index.astype(str)
    ax.plot(monthly.index, monthly.values, linewidth=2)
    ax.set_title(f"Monthly Avg: {col}")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### 3e. fact_store_operations

In [ ]:
df_store_operations["date"] = pd.to_datetime(df_store_operations["date"])
print(f"fact_store_operations: {len(df_store_operations):,} rows")
display(df_store_operations.head())
display(df_store_operations.describe().round(2))

num_cols_ops = df_store_operations.select_dtypes(include="number").columns.tolist()[:4]
fig, axes = plt.subplots(1, len(num_cols_ops), figsize=(16, 4))
if len(num_cols_ops) == 1:
    axes = [axes]
for ax, col in zip(axes, num_cols_ops):
    df_store_operations[col].hist(bins=30, ax=ax, edgecolor="white")
    ax.set_title(f"Distribution: {col}")
plt.tight_layout()
plt.show()

### 3f. fact_weather

In [ ]:
df_weather["date"] = pd.to_datetime(df_weather["date"])
print(f"fact_weather: {len(df_weather):,} rows | {df_weather['date'].min().date()} → {df_weather['date'].max().date()}")
display(df_weather.head())
display(df_weather.describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

if "temperature_c" in df_weather.columns:
    monthly_temp = df_weather.groupby(df_weather["date"].dt.to_period("M"))["temperature_c"].mean()
    monthly_temp.index = monthly_temp.index.astype(str)
    axes[0].plot(monthly_temp.index, monthly_temp.values, color="#C44E52", linewidth=2)
    axes[0].set_title("Avg Monthly Temperature (°C)")
    axes[0].tick_params(axis="x", rotation=45)

if "rainfall_mm" in df_weather.columns:
    monthly_rain = df_weather.groupby(df_weather["date"].dt.to_period("M"))["rainfall_mm"].sum()
    monthly_rain.index = monthly_rain.index.astype(str)
    axes[1].bar(monthly_rain.index, monthly_rain.values, color="#4C72B0")
    axes[1].set_title("Monthly Rainfall (mm)")
    axes[1].tick_params(axis="x", rotation=45)

if "weather_condition" in df_weather.columns:
    df_weather["weather_condition"].value_counts().plot(kind="bar", ax=axes[2], color="#55A868", rot=20)
    axes[2].set_title("Weather Condition Distribution")

plt.tight_layout()
plt.show()

### 3g. fact_restriction_events

In [ ]:
df_restrictions["date"] = pd.to_datetime(df_restrictions["date"])
print(f"fact_restriction_events: {len(df_restrictions):,} rows")
display(df_restrictions.head(20))
display(df_restrictions.describe(include="all").T)

---
## 4. Cross-Table Summary

In [ ]:
print("=" * 60)
print("SUNNYBEST SFS — RAW TABLE SUMMARY")
print("=" * 60)

print(f"\n🗂  DIMENSION TABLES")
print(f"  dim_stores          : {len(df_stores)} stores across {df_stores['city'].nunique()} cities")
print(f"  dim_products        : {len(df_products)} products, {df_products['category'].nunique()} categories, {df_products['brand'].nunique()} brands")
print(f"  dim_calendar        : {len(df_calendar)} days | {df_calendar['date'].min()} → {df_calendar['date'].max()}")
print(f"  dim_policy_regimes  : {len(df_policy_regimes)} regimes")

print(f"\n📊 FACT TABLES")
print(f"  fact_sales            : {len(df_sales):,} rows")
print(f"  fact_inventory        : {len(df_inventory):,} rows")
print(f"  fact_promotions       : {len(df_promotions):,} rows")
print(f"  fact_customer_activity: {len(df_customer_activity):,} rows")
print(f"  fact_store_operations : {len(df_store_operations):,} rows")
print(f"  fact_weather          : {len(df_weather):,} rows")
print(f"  fact_restriction_events: {len(df_restrictions):,} rows")

if "revenue" in df_sales.columns:
    print(f"\n💰 SALES")
    print(f"  Total revenue  : ₦{df_sales['revenue'].sum()/1e9:.2f}B")
    print(f"  Total units    : {df_sales['units_sold'].sum():,}")
    print(f"  Unique stores  : {df_sales['store_id'].nunique()}")
    print(f"  Unique products: {df_sales['product_id'].nunique()}")

print("\n✅ Analysis complete.")